# Stage 3 — Cap Navigation Prototype

Integrates HSV cap detection, chassis visual servoing, and yellow boundary safety into a single loop.

**States:** `SEARCH` → `APPROACH` → `COLLECT_READY` (arm grab placeholder)

**Safety priority:** boundary detection always overrides target following.

**Does NOT include:** arm collection, basket drop-off, Roboflow detection.

---
**Run order:**
1. Run all cells top-to-bottom (Shift+Enter each cell).
2. Use individual **Test** cells to validate each subsystem.
3. Run **Start Navigation** only after all tests pass.
4. Keep the **Emergency Stop** cell ready at all times.

## 1. Imports and Setup

Run once at kernel start.

In [ ]:
import cv2
import numpy as np
import time
import traitlets
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg

print('Imports OK')

## 2. Configuration

Edit these constants before each run.  
**HSV ranges MUST be calibrated on-site** using `JETANK_4_colorRecognition/colorRecognition_en.ipynb`.

In [ ]:
# ── Camera ────────────────────────────────────────────────────
CAMERA_WIDTH  = 300
CAMERA_HEIGHT = 300

# ── Motor speeds (range 0.0 to 1.0) ──────────────────────────
BASE_SPEED   = 0.30    # forward speed while approaching cap
TURN_GAIN    = 0.40    # steering gain for visual servo
SEARCH_SPEED = 0.25    # rotation speed while searching
MAX_SPEED    = 0.50    # hard clamp on any motor output

# ── Boundary avoidance ────────────────────────────────────────
AVOID_REVERSE_SPEED = 0.25    # speed while backing away from tape
AVOID_TURN_SPEED    = 0.30    # speed while turning away from tape
AVOID_REVERSE_TIME  = 0.35    # seconds to reverse
AVOID_TURN_TIME     = 0.45    # seconds to turn

# ── Detection thresholds ──────────────────────────────────────
TARGET_MIN_AREA                = 300     # blobs smaller than this are noise
TARGET_AREA_COLLECT_THRESHOLD  = 8000    # blob area that triggers COLLECT_READY
YELLOW_BOUNDARY_AREA_THRESHOLD = 2500    # yellow pixels in ROI that signal boundary

# ── Bottom ROI for boundary detection ─────────────────────────
BOTTOM_ROI_FRACTION = 0.30    # inspect bottom 30% of frame for tape

# ── Timing ────────────────────────────────────────────────────
COLLECT_READY_PAUSE = 1.5     # seconds robot pauses at COLLECT_READY

# ── HSV: Bottle caps ──────────────────────────────────────────
# CALIBRATE ON-SITE before every run using JETANK_4_colorRecognition.
# Default below assumes yellow caps. Change H/S/V to match real cap color.
CAP_HSV_LOWER = np.array([24, 100, 100])
CAP_HSV_UPPER = np.array([44, 255, 255])

# ── HSV: Yellow boundary tape ─────────────────────────────────
# Slightly wider range than cap HSV to handle varying tape lighting.
# If caps and tape share color, distinguish by area threshold in ROI.
TAPE_HSV_LOWER = np.array([15,  60,  60])
TAPE_HSV_UPPER = np.array([45, 255, 255])

print('Configuration loaded.')
print('  Camera     :', CAMERA_WIDTH, 'x', CAMERA_HEIGHT)
print('  Base speed :', BASE_SPEED, '  Max speed:', MAX_SPEED)
print('  Cap HSV    :', CAP_HSV_LOWER, '->', CAP_HSV_UPPER)
print('  Tape HSV   :', TAPE_HSV_LOWER, '->', TAPE_HSV_UPPER)

## 3. Camera and Robot Setup

In [ ]:
camera = Camera.instance(width=CAMERA_WIDTH, height=CAMERA_HEIGHT)

# Debug display widgets — updated on every frame by execute()
image_widget = widgets.Image(format='jpeg', width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
mask_widget  = widgets.Image(format='jpeg', width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
state_label  = widgets.Label(value='State: STOPPED')
area_label   = widgets.Label(value='Cap area: 0')
yellow_label = widgets.Label(value='Yellow area: 0')
motor_label  = widgets.Label(value='Motors: L=0.00  R=0.00')

display(widgets.VBox([
    widgets.HBox([image_widget, mask_widget]),
    widgets.VBox([state_label, area_label, yellow_label, motor_label])
]))

print('Camera ready.')

In [ ]:
robot = Robot()
robot.stop()
print('Robot initialized and stopped.')

## 4. HSV Cap Detection

Adapted from `JETANK_5_colorTracking/colorTracking_en.ipynb`.  
Returns the largest blob matching `CAP_HSV_LOWER`/`CAP_HSV_UPPER`.

In [ ]:
def detect_cap(frame):
    """
    Find the largest cap-colored blob in frame.
    Returns dict: found, center_x, center_y, radius, area, mask.
    """
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, CAP_HSV_LOWER, CAP_HSV_UPPER)
    mask = cv2.erode(mask,  None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)

    # [-2] handles both OpenCV 3.x (returns 3 values) and 4.x (returns 2 values)
    cnts = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL,
                            cv2.CHAIN_APPROX_SIMPLE)[-2]

    empty = {'found': False, 'center_x': 0, 'center_y': 0,
             'radius': 0, 'area': 0, 'mask': mask}

    if not cnts:
        return empty

    c    = max(cnts, key=cv2.contourArea)
    area = int(cv2.contourArea(c))

    if area < TARGET_MIN_AREA:
        return empty

    ((cx, cy), radius) = cv2.minEnclosingCircle(c)
    return {
        'found':    True,
        'center_x': int(cx),
        'center_y': int(cy),
        'radius':   int(radius),
        'area':     area,
        'mask':     mask,
    }

print('detect_cap() defined.')

## 5. Yellow Boundary Detection

Only inspects the bottom `BOTTOM_ROI_FRACTION` of the frame — the zone where immediate
boundary danger is visible. Also returns left/right split to guide avoidance direction.

In [ ]:
def detect_yellow_boundary(frame):
    """
    Check bottom ROI for yellow tape pixels.
    Returns dict: detected, yellow_area, yellow_left, yellow_right, yellow_mask.
    """
    h         = frame.shape[0]
    roi_start = int(h * (1.0 - BOTTOM_ROI_FRACTION))
    roi       = frame[roi_start:, :]

    hsv_roi  = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    roi_mask = cv2.inRange(hsv_roi, TAPE_HSV_LOWER, TAPE_HSV_UPPER)

    yellow_area  = int(np.sum(roi_mask > 0))
    mid          = roi_mask.shape[1] // 2
    yellow_left  = int(np.sum(roi_mask[:, :mid] > 0))
    yellow_right = int(np.sum(roi_mask[:, mid:] > 0))

    # Full-frame mask for display (top portion is black)
    full_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    full_mask[roi_start:, :] = roi_mask

    return {
        'detected':     yellow_area >= YELLOW_BOUNDARY_AREA_THRESHOLD,
        'yellow_area':  yellow_area,
        'yellow_left':  yellow_left,
        'yellow_right': yellow_right,
        'yellow_mask':  full_mask,
    }

print('detect_yellow_boundary() defined.')

## 6. Debug Visualization

In [ ]:
def draw_debug(frame, cap, boundary, state, left_spd, right_spd):
    """Return annotated copy of frame with detection results and state overlay."""
    out = frame.copy()

    # Cap bounding circle
    if cap['found']:
        cx, cy, r = cap['center_x'], cap['center_y'], cap['radius']
        cv2.circle(out, (cx, cy), r, (0, 255, 0), 2)
        cv2.circle(out, (cx, cy), 5, (0, 255, 0), -1)
        cv2.putText(out, 'a=' + str(cap['area']), (cx - 20, cy - r - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

    # ROI divider — red if boundary detected, yellow otherwise
    roi_y      = int(frame.shape[0] * (1.0 - BOTTOM_ROI_FRACTION))
    line_color = (0, 0, 255) if boundary['detected'] else (0, 200, 200)
    cv2.line(out, (0, roi_y), (frame.shape[1], roi_y), line_color, 2)

    # State and motor overlay
    cv2.putText(out, 'State: ' + state, (5, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(out,
                'L=' + str(round(left_spd, 2)) + ' R=' + str(round(right_spd, 2)),
                (5, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    if boundary['detected']:
        cv2.putText(out, 'BOUNDARY!', (5, 65),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2)

    return out

print('draw_debug() defined.')

## 7. State Machine

```
SEARCH        rotate slowly, scan for caps
  |-- cap found --> APPROACH

APPROACH      visual-servo toward cap
  |-- cap lost  --> SEARCH
  |-- area >= threshold --> COLLECT_READY
  |-- boundary detected --> BOUNDARY_AVOID

COLLECT_READY stop, log, pause 1.5s (arm grab goes here in Stage 4)
  |-- timeout --> SEARCH

BOUNDARY_AVOID reverse 0.35s, turn 0.45s
  |-- done --> SEARCH
```

Boundary avoidance has **highest priority** over all other states except `COLLECT_READY`  
(robot is already stopped when collecting).

In [ ]:
# Global state — reset this cell to restart the state machine
nav_state          = 'SEARCH'
avoid_start_time   = 0.0
avoid_turn_right   = True     # direction chosen at boundary detection
collect_start_time = 0.0
collect_logged     = False
last_left_spd      = 0.0
last_right_spd     = 0.0

print('State machine initialized. State: SEARCH')

In [ ]:
def execute(change):
    """Per-frame callback — runs detection, state machine, and motor commands."""
    global nav_state, avoid_start_time, avoid_turn_right
    global collect_start_time, collect_logged
    global last_left_spd, last_right_spd

    frame     = change['new']
    left_spd  = 0.0
    right_spd = 0.0

    try:
        # ── Detectors ────────────────────────────────────────────
        boundary = detect_yellow_boundary(frame)
        cap      = detect_cap(frame)

        # ── State machine ─────────────────────────────────────────

        # Priority 1: continue or enter boundary avoidance
        if nav_state == 'BOUNDARY_AVOID':
            elapsed = time.time() - avoid_start_time
            if elapsed < AVOID_REVERSE_TIME:
                # Phase 1: back away from tape
                left_spd  = -AVOID_REVERSE_SPEED
                right_spd = -AVOID_REVERSE_SPEED
            elif elapsed < AVOID_REVERSE_TIME + AVOID_TURN_TIME:
                # Phase 2: turn away from the tape side
                if avoid_turn_right:
                    left_spd  =  AVOID_TURN_SPEED
                    right_spd = -AVOID_TURN_SPEED
                else:
                    left_spd  = -AVOID_TURN_SPEED
                    right_spd =  AVOID_TURN_SPEED
            else:
                # Phase 3: avoidance complete, return to search
                nav_state = 'SEARCH'
                left_spd  = 0.0
                right_spd = 0.0

        elif boundary['detected'] and nav_state != 'COLLECT_READY':
            # Trigger avoidance from SEARCH or APPROACH
            # Turn away from whichever side has more yellow
            avoid_start_time = time.time()
            avoid_turn_right = boundary['yellow_left'] >= boundary['yellow_right']
            nav_state        = 'BOUNDARY_AVOID'
            left_spd         = -AVOID_REVERSE_SPEED
            right_spd        = -AVOID_REVERSE_SPEED

        else:
            # Priority 2: normal navigation states

            # Promote SEARCH -> APPROACH the first frame a cap is seen
            if nav_state == 'SEARCH' and cap['found']:
                nav_state = 'APPROACH'

            if nav_state == 'SEARCH':
                # Rotate slowly left to scan
                left_spd  =  SEARCH_SPEED
                right_spd = -SEARCH_SPEED

            elif nav_state == 'APPROACH':
                if not cap['found']:
                    # Lost the cap — go back to searching
                    nav_state = 'SEARCH'
                    left_spd  =  SEARCH_SPEED
                    right_spd = -SEARCH_SPEED
                elif cap['area'] >= TARGET_AREA_COLLECT_THRESHOLD:
                    # Close enough — stop and signal collect
                    nav_state          = 'COLLECT_READY'
                    collect_start_time = time.time()
                    collect_logged     = False
                    left_spd           = 0.0
                    right_spd          = 0.0
                else:
                    # Visual servo: steer toward cap center
                    # error_x_norm in range -1.0 (left) to +1.0 (right)
                    err       = (cap['center_x'] - CAMERA_WIDTH / 2.0) / (CAMERA_WIDTH / 2.0)
                    left_spd  = max(-MAX_SPEED, min(MAX_SPEED, BASE_SPEED + TURN_GAIN * err))
                    right_spd = max(-MAX_SPEED, min(MAX_SPEED, BASE_SPEED - TURN_GAIN * err))

            elif nav_state == 'COLLECT_READY':
                left_spd  = 0.0
                right_spd = 0.0
                if not collect_logged:
                    area_val = str(cap['area']) if cap['found'] else 'n/a'
                    print('[COLLECT_READY] Target reached. Area=' + area_val +
                          '. Stage 4 arm sequence goes here.')
                    collect_logged = True
                # Return to SEARCH after pause (Stage 4: arm grab replaces this delay)
                if time.time() - collect_start_time >= COLLECT_READY_PAUSE:
                    collect_start_time = 0.0
                    collect_logged     = False
                    nav_state          = 'SEARCH'

        # ── Drive motors ──────────────────────────────────────────
        robot.left_motor.value  = left_spd
        robot.right_motor.value = right_spd
        last_left_spd  = left_spd
        last_right_spd = right_spd

        # ── Update debug widgets ──────────────────────────────────
        annotated          = draw_debug(frame, cap, boundary, nav_state, left_spd, right_spd)
        image_widget.value = bgr8_to_jpeg(annotated)

        # Mask widget: cap mask in white, tape detection in cyan
        mask_bgr = cv2.cvtColor(cap['mask'], cv2.COLOR_GRAY2BGR)
        ym = boundary['yellow_mask']
        mask_bgr[ym > 0] = [0, 255, 255]
        mask_widget.value = bgr8_to_jpeg(mask_bgr)

        state_label.value  = 'State: ' + nav_state
        area_val = str(cap['area']) if cap['found'] else '0'
        area_label.value   = ('Cap area: ' + area_val +
                              '  (collect @ ' + str(TARGET_AREA_COLLECT_THRESHOLD) + ')')
        yellow_label.value = ('Yellow: ' + str(boundary['yellow_area']) +
                              '  L=' + str(boundary['yellow_left']) +
                              '  R=' + str(boundary['yellow_right']))
        motor_label.value  = ('Motors L=' + str(round(left_spd, 3)) +
                              '  R=' + str(round(right_spd, 3)))

    except Exception as e:
        robot.stop()
        print('[ERROR] execute() failed: ' + str(e))
        raise

print('execute() defined.')

## 8. Start Navigation

Run this cell to begin the autonomous loop.  
**Robot will move.** Ensure at least 1 m clearance on all sides.

Run the **Emergency Stop** cell below at any time to halt.

In [ ]:
camera.unobserve_all()
camera.observe(execute, names='value')
print('Navigation running. State=' + nav_state)
print('Run Emergency Stop cell to halt.')

## 9. Emergency Stop

Run this cell at any time to immediately halt all motion.

In [ ]:
camera.unobserve_all()
robot.stop()
print('STOPPED. Motors off. Camera unlinked.')

## 10. Reset State Machine

Run to restart from SEARCH without restarting the kernel.

In [ ]:
camera.unobserve_all()
robot.stop()

nav_state          = 'SEARCH'
avoid_start_time   = 0.0
avoid_turn_right   = True
collect_start_time = 0.0
collect_logged     = False
last_left_spd      = 0.0
last_right_spd     = 0.0

print('State machine reset to SEARCH. Robot stopped.')

---
## Test Procedures

Run tests in order. Do NOT run Start Navigation until all tests pass.

| # | Test | Motors move? | What to check |
|---|------|-------------|---------------|
| 1 | Yellow boundary detection | No | Yellow area > threshold when tape is close |
| 2 | Cap detection | No | Cap found, center and area reported correctly |
| 3 | Search rotation | Yes (2 s) | Robot rotates smoothly and stops |
| 4 | Boundary avoidance | Yes | Robot backs up and turns when tape is shown |
| 5 | Full SEARCH → APPROACH → COLLECT_READY | Yes | Robot tracks cap and stops near it |

### Test 1 — Yellow Boundary Detection (no motors)

Point the camera at yellow tape (or hold tape below the camera).  
Run this cell and check that `detected` becomes `True`.

If `yellow_area` stays at 0: adjust `TAPE_HSV_LOWER`/`TAPE_HSV_UPPER` in the config cell.

In [ ]:
print('TEST 1 — Yellow Boundary Detection')
print('(No motors. Safe to run at any time.)')
print()

frame  = camera.value.copy()
result = detect_yellow_boundary(frame)

print('Yellow area  :', result['yellow_area'])
print('Threshold    :', YELLOW_BOUNDARY_AREA_THRESHOLD)
print('Detected     :', result['detected'])
print('Left half    :', result['yellow_left'])
print('Right half   :', result['yellow_right'])
print()

# Visualize: draw ROI line and highlight tape pixels in cyan
vis   = frame.copy()
roi_y = int(frame.shape[0] * (1.0 - BOTTOM_ROI_FRACTION))
cv2.line(vis, (0, roi_y), (frame.shape[1], roi_y), (0, 255, 0), 2)
ym = result['yellow_mask']
vis[ym > 0] = [0, 255, 255]

display(widgets.Image(value=bgr8_to_jpeg(vis), format='jpeg',
                      width=CAMERA_WIDTH, height=CAMERA_HEIGHT))

### Test 2 — Cap Detection (no motors)

Place a bottle cap in front of the camera at ~0.5 m distance.  
Run this cell and check that the cap is detected and the circle overlay is correct.

If detection fails: adjust `CAP_HSV_LOWER`/`CAP_HSV_UPPER` in the config cell using  
`JETANK_4_colorRecognition/colorRecognition_en.ipynb` to sample the actual HSV values.

In [ ]:
print('TEST 2 — Cap Detection')
print('(No motors. Safe to run at any time.)')
print()

frame = camera.value.copy()
cap   = detect_cap(frame)

print('Cap found  :', cap['found'])
if cap['found']:
    print('Center     :', cap['center_x'], ',', cap['center_y'])
    print('Radius     :', cap['radius'])
    print('Area       :', cap['area'])
    print('Collect at :', TARGET_AREA_COLLECT_THRESHOLD)
    closeness = round(100.0 * cap['area'] / TARGET_AREA_COLLECT_THRESHOLD, 1)
    print('Closeness  :', closeness, '%  (100% = trigger COLLECT_READY)')
else:
    print('No cap detected.')
    print('If a cap is visible, adjust CAP_HSV_LOWER / CAP_HSV_UPPER.')
print()

# Draw detection
vis = frame.copy()
if cap['found']:
    cv2.circle(vis, (cap['center_x'], cap['center_y']), cap['radius'], (0, 255, 0), 2)
    cv2.circle(vis, (cap['center_x'], cap['center_y']), 5, (0, 255, 0), -1)

display(widgets.Image(value=bgr8_to_jpeg(vis), format='jpeg',
                      width=CAMERA_WIDTH, height=CAMERA_HEIGHT))

# Show detection mask
mask_display = cv2.cvtColor(cap['mask'], cv2.COLOR_GRAY2BGR)
display(widgets.Image(value=bgr8_to_jpeg(mask_display), format='jpeg',
                      width=CAMERA_WIDTH, height=CAMERA_HEIGHT))

### Test 3 — Search Rotation (motors move for 2 seconds)

**Robot will rotate left for 2 seconds then stop.**  
Ensure at least 0.5 m of clearance around the robot before running.

In [ ]:
print('TEST 3 — Search rotation for 2 seconds.')
print('Robot rotating LEFT at speed', SEARCH_SPEED)

robot.left_motor.value  =  SEARCH_SPEED
robot.right_motor.value = -SEARCH_SPEED
time.sleep(2.0)
robot.stop()

print('Done. Motors stopped.')

### Test 4 — Boundary Avoidance (motors move)

1. Run the **Start Navigation** cell (Section 8).
2. Slide a piece of yellow tape under the front of the camera slowly.
3. Observe: robot should stop, reverse briefly, turn away, then return to SEARCH.
4. Check that `State: BOUNDARY_AVOID` appears in the debug display.
5. Run **Emergency Stop** when done.

**Tuning:** If the robot does not react, decrease `YELLOW_BOUNDARY_AREA_THRESHOLD` in the config cell.  
If it reacts to the floor color, increase the threshold or narrow `TAPE_HSV_LOWER`/`TAPE_HSV_UPPER`.

### Test 5 — Full SEARCH → APPROACH → COLLECT_READY

1. Place a bottle cap on the floor ~1 m in front of the robot.
2. Run the **Start Navigation** cell (Section 8).
3. Observe state transitions in the debug display:
   - `SEARCH` → robot rotates until cap is in frame
   - `APPROACH` → robot drives toward cap, steering to center it
   - `COLLECT_READY` → robot stops, prints log message, then returns to SEARCH
4. Confirm that `[COLLECT_READY]` appears in the cell output.
5. Run **Emergency Stop** when done.

**Tuning notes:**
- If robot overshoots: reduce `TURN_GAIN`
- If robot is too slow to steer: increase `TURN_GAIN`
- If COLLECT_READY triggers too far away: increase `TARGET_AREA_COLLECT_THRESHOLD`
- If robot never reaches COLLECT_READY: decrease `TARGET_AREA_COLLECT_THRESHOLD`

---
## Notes for Stage 4

The `COLLECT_READY` state is a placeholder. In Stage 4, replace the `COLLECT_READY_PAUSE` delay with:

```python
# Stage 4: arm grab sequence (not implemented here)
TTLServo.xyInputSmooth(grab_x, grab_y, t)   # lower arm to grab pose
time.sleep(t)
TTLServo.servoAngleCtrl(4, -90, 1, 150)     # close gripper
time.sleep(0.5)
TTLServo.xyInputSmooth(carry_x, carry_y, t) # raise arm to carry pose
```

After successful collection, add a `DEPOSIT` state that drives open-loop to the known basket position.